In [24]:
import pandas as pd
# OCR
import pytesseract
pytesseract.pytesseract.tesseract_cmd = '/opt/homebrew/bin/tesseract'  # Path to tesseract executable  # For Apple Silicon Macs
from PIL import Image
import re # Regular expressions
import os

In [21]:
folder_path = 'data/transaction/dime/mutual_funds/'

image = Image.open('data/transaction/dime/mutual_funds/20240527_B_K-CHINA-A(D).jpeg')

# Perform OCR
text = pytesseract.image_to_string(image, lang='eng')

print(text)

Status (As of 27 May 2024 - 12:30)

@ To Execute Order

Your order will be submitted to
the AMC at the fund's cut off
time. You can cancel it before
then.

Buy K-CHINA-A(D)

1,000.00 THB

Submission Date 27 May 2024 - 12:29
Payment Date 27 May 2024 - 12:29
Effective Date 28 May 2024
Dime! Portfolio Mutual Fund Port
Unitholder No. OMN
Account No. 101710737908447
Order ID 2202405270009583
Reference ID DIMEPRSUB2705202405295

74810160

You can cancel this order before 15:30, 28 May 2024.



In [22]:
# Regular expressions to extract data
status_pattern = re.search(r'@ (.*?)\n', text)
status = status_pattern.group(1) if status_pattern else None

order_type_pattern = re.search(r'@ (.*?)\n', text)
order_type = order_type_pattern.group(1) if order_type_pattern else None

order_pattern = re.search(r'(Buy|Sell) (.*?)\n', text)
position, ticker = (order_pattern.group(1), order_pattern.group(2)) if order_pattern else (None, None)

thb_amount_pattern = re.search(r'(\d+,\d+.\d+) THB\n', text)
thb_amount = thb_amount_pattern.group(1) if thb_amount_pattern else None

submission_date_pattern = re.search(r'Submission Date (.*?)\n', text)
submission_date = submission_date_pattern.group(1) if submission_date_pattern else None

payment_date_pattern = re.search(r'Payment Date (.*?)\n', text)
payment_date = payment_date_pattern.group(1) if payment_date_pattern else None

effective_date_pattern = re.search(r'Effective Date (.*?)\n', text)
effective_date = effective_date_pattern.group(1) if effective_date_pattern else None

portfolio_pattern = re.search(r'Dime! Portfolio (.*?)\n', text)
portfolio = portfolio_pattern.group(1) if portfolio_pattern else None

unitholder_no_pattern = re.search(r'Unitholder No. (.*?)\n', text)
unitholder_no = unitholder_no_pattern.group(1) if unitholder_no_pattern else None

account_no_pattern = re.search(r'Account No. (.*?)\n', text)
account_no = account_no_pattern.group(1) if account_no_pattern else None

order_id_pattern = re.search(r'Order ID (.*?)\n', text)
order_id = order_id_pattern.group(1) if order_id_pattern else None

reference_id_pattern = re.search(r'Reference ID (.*?)\n\n(\d+)', text)
reference_id = reference_id_pattern.group(1) + reference_id_pattern.group(2) if reference_id_pattern else None

# Organize extracted data into a dictionary
data = {
    "Status": [status],
    "Position": [position],
    "Ticker": [ticker],
    "Amount (THB)": [thb_amount],
    "Submission Date": [submission_date],
    "Payment Date": [payment_date],
    "Effective Date": [effective_date],
    "Portfolio": [portfolio],
    "Unitholder No.": [unitholder_no],
    "Account No.": [account_no],
    "Order ID": [order_id],
    "Reference ID": [reference_id]
}

# convert to datetime
data['Submission Date'] = pd.to_datetime(data['Submission Date'])
data['Payment Date'] = pd.to_datetime(data['Payment Date'])
data['Effective Date'] = pd.to_datetime(data['Effective Date'])


df = pd.DataFrame(data)

df

,Status,Position,Ticker,Amount (THB),Submission Date,Payment Date,Effective Date,Portfolio,Unitholder No.,Account No.,Order ID,Reference ID
0,To Execute Order,Buy,K-CHINA-A(D),"1,000.00",2024-05-27 12:29:00,2024-05-27 12:29:00,2024-05-28,Mutual Fund Port,OMN,101710737908447,2202405270009583,DIMEPRSUB270520240529574810160


In [ ]:
# Folder containing the images
folder_path = 'data/transaction/dime/mutual_funds/'

# List to store data for each image
data_list = []

# Function to extract data from text
def extract_data_from_text(text):
    status_pattern = re.search(r'@ (.*?)\n', text)
    status = status_pattern.group(1) if status_pattern else None

    order_type_pattern = re.search(r'@ (.*?)\n', text)
    order_type = order_type_pattern.group(1) if order_type_pattern else None

    order_pattern = re.search(r'(Buy|Sell) (.*?)\n', text)
    position, ticker = (order_pattern.group(1), order_pattern.group(2)) if order_pattern else (None, None)

    thb_amount_pattern = re.search(r'(\d+,\d+.\d+) THB\n', text)
    thb_amount = thb_amount_pattern.group(1) if thb_amount_pattern else None

    submission_date_pattern = re.search(r'Submission Date (.*?)\n', text)
    submission_date = submission_date_pattern.group(1) if submission_date_pattern else None

    payment_date_pattern = re.search(r'Payment Date (.*?)\n', text)
    payment_date = payment_date_pattern.group(1) if payment_date_pattern else None

    effective_date_pattern = re.search(r'Effective Date (.*?)\n', text)
    effective_date = effective_date_pattern.group(1) if effective_date_pattern else None

    portfolio_pattern = re.search(r'Dime! Portfolio (.*?)\n', text)
    portfolio = portfolio_pattern.group(1) if portfolio_pattern else None

    unitholder_no_pattern = re.search(r'Unitholder No. (.*?)\n', text)
    unitholder_no = unitholder_no_pattern.group(1) if unitholder_no_pattern else None

    account_no_pattern = re.search(r'Account No. (.*?)\n', text)
    account_no = account_no_pattern.group(1) if account_no_pattern else None

    order_id_pattern = re.search(r'Order ID (.*?)\n', text)
    order_id = order_id_pattern.group(1) if order_id_pattern else None

    reference_id_pattern = re.search(r'Reference ID (.*?)\n\n(\d+)', text)
    reference_id = reference_id_pattern.group(1) + reference_id_pattern.group(2) if reference_id_pattern else None

    return {
        "Status": status,
        "Position": position,
        "Ticker": ticker,
        "Amount (THB)": thb_amount,
        "Submission Date": submission_date,
        "Payment Date": payment_date,
        "Effective Date": effective_date,
        "Portfolio": portfolio,
        "Unitholder No.": unitholder_no,
        "Account No.": account_no,
        "Order ID": order_id,
        "Reference ID": reference_id
    }

# Iterate over all files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith(".jpeg") or filename.endswith(".jpg") or filename.endswith(".png"):
        file_path = os.path.join(folder_path, filename)

        # Open the image file
        image = Image.open(file_path)

        # Perform OCR
        text = pytesseract.image_to_string(image, lang='eng')

        # Extract data from text
        data = extract_data_from_text(text)

        # Append data to the list
        data_list.append(data)

# Create DataFrame from the list
df = pd.DataFrame(data_list)

# Convert date columns to datetime
df['Submission Date'] = pd.to_datetime(df['Submission Date'], errors='coerce')
df['Payment Date'] = pd.to_datetime(df['Payment Date'], errors='coerce')
df['Effective Date'] = pd.to_datetime(df['Effective Date'], errors='coerce')

# Display DataFrame
print(df)